# Road Crack Detection — Prediction with `best.pt`

This notebook is for **running predictions only** with your already-trained YOLO11 model. It does **not** retrain the model, resume training, or require a GPU.

**Chosen confidence threshold:** `0.45` — your previous validation experiment measured 87.59% precision and 69.82% recall at this setting. Results on new images may differ.

### Important: Google Drive mounting failed earlier
This notebook does **not** require `drive.mount()`.
- First, make sure `best.pt` is downloaded to your computer.
- Upload `best.pt` in Cell 3 when prompted.
- Then upload a road image in Cell 4.

Use `best.pt` for prediction. Keep `last.pt` as a backup for possible future training resumption.


## Cell 1 — Install/import Ultralytics

In [ ]:
%pip -q install ultralytics

from ultralytics import YOLO
from google.colab import files
from IPython.display import display
from PIL import Image
from pathlib import Path
import cv2
import os

print("Ultralytics is ready.")


## Cell 2 — Set the confidence threshold

In [ ]:
CONF_THRESHOLD = 0.45
print("Prediction confidence threshold:", CONF_THRESHOLD)


## Cell 3 — Upload and load your trained `best.pt`

In [ ]:
print("Select your trained best.pt file from your computer.")
model_upload = files.upload()

pt_files = [name for name in model_upload if name.lower().endswith(".pt")]
if not pt_files:
    raise ValueError("No .pt file was uploaded. Please upload best.pt.")

# Prefer a file named best.pt if multiple .pt files were selected.
model_file = next((name for name in pt_files if Path(name).name.lower() == "best.pt"), pt_files[0])
MODEL_PATH = str(Path("/content") / Path(model_file).name)

print("Loading model from:", MODEL_PATH)
model = YOLO(MODEL_PATH)
print("Model loaded. Classes:", model.names)
print("GPU available to PyTorch will be used automatically if supported; otherwise prediction runs on CPU.")


## Cell 4 — Upload a road image and predict

In [ ]:
print("Select a road image (JPG, JPEG, PNG, or WEBP).")
image_upload = files.upload()

image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
image_files = [name for name in image_upload if Path(name).suffix.lower() in image_extensions]
if not image_files:
    raise ValueError("No supported image was uploaded. Upload a JPG or PNG image.")

IMAGE_PATH = image_files[0]
print("Image:", IMAGE_PATH)

results = model.predict(
    source=IMAGE_PATH,
    conf=CONF_THRESHOLD,
    save=False,
    verbose=False
)
result = results[0]
print("Detections retained at conf=", CONF_THRESHOLD, ": ", len(result.boxes), sep="")

# Ultralytics result.plot() returns a BGR image; convert to RGB for display.
annotated_bgr = result.plot()
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
output_path = "/content/road_crack_prediction.jpg"
Image.fromarray(annotated_rgb).save(output_path, quality=95)
display(Image.open(output_path))
print("Saved prediction image to:", output_path)


## Cell 5 — Download the prediction image

In [ ]:
files.download("/content/road_crack_prediction.jpg")


## Notes
- `conf=0.45` filters predictions; it does not change or retrain the model weights.
- Your previously measured precision is a validation result, not a guarantee for every new image.
- If `best.pt` is only in Google Drive and not on your computer, open Google Drive in your browser, download `best.pt` manually, then upload it in Cell 3. This avoids relying on the Drive mount that failed earlier.
- If you later retrain, save checkpoints to persistent storage and keep both `best.pt` and `last.pt`.
